# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's daily search performance for one client, identified by **client_hash_id**, **content_hash_id**, and **report_date**. I'm using **fact_content_daily_performance** and joining **dim_content** for static content attributes. I use the March 2026 mid-panel period for development and keep June 2026 as the sealed test month.

In [1]:
import os, getpass, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
dim_clients = f"read_parquet('{REL}/dim_clients.parquet')"

ANCHOR = "DATE '2026-03-31'"  # decision moment: end of a mid-panel month

HF token: ··········


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Feature:** **imp_prev30**, **clk_prev30**, **pos_prev30**, **ctr_prev30** (all aggregated over days 31–60 before the anchor), **content_age_days** (from **dim_content**, static and known well before the anchor).
* **Label/proxy:** **is_declining** = impressions in the last 30 days fell more than 20% vs. the prior 30 days. Computed only from gsc_impressions.
* **Context:** **client_hash_id**, **content_hash_id**, **report_date**, used only for grouping and joining.
* **Excluded:** **fact_content_query_90d**'s columns. Its 90-day window is fixed and overlaps the last-30-day label window, so any feature from it right now would be a leak risk until the windows are properly aligned. Also excluding raw GA4 columns, **ga4_data_available** is FALSE for a meaningful share of rows and those aren't real zeros.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1, grain:**

In [2]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {fact}
    WHERE report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").df()
print(f"Grain violations: {len(grain_check)}")  # expect 0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations: 0


**Query 2, row count + date span:**

In [3]:
span = con.sql(f"""
    SELECT COUNT(*) n_rows, MIN(report_date) d_min, MAX(report_date) d_max,
           COUNT(DISTINCT content_hash_id) n_content, COUNT(DISTINCT client_hash_id) n_clients
    FROM {fact}
    WHERE report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
""").df()
print(span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     n_rows      d_min      d_max  n_content  n_clients
0  17717009 2026-01-30 2026-03-31     349411         59


**Query 3, availability:**

In [4]:
avail = con.sql(f"""
    SELECT COUNT(*) total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) ga4_ok
    FROM {fact}
    WHERE report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
""").df()
print(avail)
print(f"GA4-available share: {avail['ga4_ok'][0] / avail['total'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      total    ga4_ok
0  17717009  567080.0
GA4-available share: 3.2%


**Five-feature frame:**

In [12]:
features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date >  {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END)        AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2
    HAVING imp_prev30 >= 100
""").df()

features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

features = features.merge(
    con.sql(f"""
        SELECT content_hash_id,
               DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
        FROM {dim_content}
    """).df(),
    on='content_hash_id', how='left'
)

features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
print(f"{len(features):,} content items")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82,549 content items


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30,ctr_prev30,content_age_days,is_declining
0,client_e547b89c05043229,content_745892967f4717c8,2123.0,2311.0,9.0,6.700516,0.003894,230,0
1,client_e547b89c05043229,content_f74790c6cb61ec73,1076.0,623.0,2.0,12.169141,0.003210,230,0
2,client_e547b89c05043229,content_5d91a9cf8b010fc7,573.0,303.0,0.0,25.778184,0.000000,230,0
3,client_e547b89c05043229,content_5d8a00c0fbd256e2,495.0,318.0,0.0,34.243901,0.000000,230,0
4,client_e547b89c05043229,content_1f6c245ff7e794fc,4412.0,1451.0,0.0,26.588076,0.000000,230,0


* **imp_prev30**: knowable at the decision moment because it only sums impressions from before the label's 30-day outcome window.
* **clk_prev30**: same reasoning, same prior window.
* **pos_prev30**: same prior window, describes ranking behavior before the decision.
* **ctr_prev30**: derived only from the two columns above, so it inherits their safety.
* **content_age_days**: a static content attribute set at creation, not affected by future performance.

**The trap:**

In [10]:
from sklearn.metrics import roc_auc_score

# Deliberately smuggle in a label-derived column
features['leaky_imp_last30'] = features['imp_last30']  # this IS the label's own numerator

honest_score = roc_auc_score(features['is_declining'], features['imp_prev30'])
leaked_score = roc_auc_score(features['is_declining'], -features['leaky_imp_last30'])
print(f"Honest single-feature AUC (imp_prev30):   {honest_score:.3f}")
print(f"Leaked single-feature AUC (imp_last30):   {leaked_score:.3f}")

features = features.drop(columns=['leaky_imp_last30'])  # remove it, keep the honest number

Honest single-feature AUC (imp_prev30):   0.450
Leaked single-feature AUC (imp_last30):   0.762


The leaked feature increases the single-feature AUC from 0.450 to 0.762 because **imp_last30** is directly used to construct the **is_declining** label. It therefore contains information from the outcome window that would not be available at the decision moment. I remove this feature and keep the honest AUC of 0.450.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation is that I cannot safely use **fact_content_query_90d** for query-level features yet. Its fixed 90-day window overlaps the label window, so using it without a way to separate the periods could introduce leakage. This means query-level features are currently excluded from this slice.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.